In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn import tree
from sklearn.metrics import f1_score


In [2]:
#Data Preparation
df = pd.read_csv('hotel_bookings.csv', na_values=['NULL', 'NaN'])

#Usunięcie duplikatów

initial_len = len(df)
df_clean = df.drop_duplicates().copy()
print(f"Usunięto {initial_len - len(df_clean)} duplikatów.")

#Uzupełnienie brakujących danych
df_clean['children'] = df_clean['children'].fillna(0)
df_clean['agent'] = df_clean['agent'].fillna(0)
df_clean['country'] = df_clean['country'].fillna('Unknown')  # Zatrzymujemy kolumnę!

#Usunięcie kolumny company
df_clean.drop(columns=['company'], inplace=True, errors='ignore')

#Usunięcie rekordów z błędnym ADR i rezerwacji z brakiem gości
df_clean = df_clean[(df_clean['adr'] >= 0) & (df_clean['adr'] < 1000)]
df_clean = df_clean[(df_clean['adults'] + df_clean['children'] + df_clean['babies']) > 0]

#Usunięcie wycieków danych
leaked_cols = ['reservation_status', 'reservation_status_date', 'assigned_room_type']
df_clean.drop(columns=leaked_cols, inplace=True, errors='ignore')

#Weryfikacja końcowa
print(f"Podsumowanie czyszczenia")
print(f"Pozostało wierszy: {len(df_clean)}")
print(f"Brakujące wartości po czyszczeniu: {df_clean.isnull().sum().sum()}")

Usunięto 31994 duplikatów.
Podsumowanie czyszczenia
Pozostało wierszy: 87228
Brakujące wartości po czyszczeniu: 0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# Zmiana tekstu na liczby + chronologia
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_clean['arrival_date_month'] = df_clean['arrival_date_month'].map(month_map)

# Inżynieria nowych cech
df_clean['total_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']
df_clean['total_guests'] = df_clean['adults'] + df_clean['children'] + df_clean['babies']

# Usunięcie zmiennej country
if 'country' in df_clean.columns:
    df_clean = df_clean.drop(columns=['country'])

# Ordinal Encoding zamiast One-Hot Encoding
cols_to_encode = ['hotel', 'meal', 'market_segment',
                  'distribution_channel', 'reserved_room_type',
                  'deposit_type', 'customer_type']

encoder = OrdinalEncoder()
# Transformacja w miejscu - tekst zamienia się na liczby zmiennoprzecinkowe (0.0, 1.0, 2.0...)
df_clean[cols_to_encode] = encoder.fit_transform(df_clean[cols_to_encode])

df_final = df_clean.copy()

# Weryfikacja końcowa
print(f"Zakończono! Liczba wierszy: {df_final.shape[0]}, Liczba kolumn: {df_final.shape[1]}")

#Przygotowanie zbioru danych do dalszej pracy

#Wyodrębnienie X oraz y
X = df_final.drop(columns=['is_canceled', 'country'], errors='ignore')
y = df_final['is_canceled']

#Podział na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Zakończono! Liczba wierszy: 87228, Liczba kolumn: 29


In [4]:
# Cut the first tree using CCP_alpha
depths_to_test = [5,6,7,8,9,10,11,12,13,14,15,16,17,18]

for depth in depths_to_test:
    print(f"\n--- Rozpoczynam test dla max_depth = {depth} ---")

    # 1. Start z bazowym, ograniczonym drzewem
    base_tree = tree.DecisionTreeClassifier(random_state=12345, max_depth=depth)

    # 2. Wyciągnięcie ścieżki CCP dla ZBIORU 2 (Ordinal Encoding)
    path = base_tree.cost_complexity_pruning_path(X_train, y_train)
    ccp_alphas = path.ccp_alphas



    # 3. Pętla trenująca
    trees = []
    for ccp_alpha in ccp_alphas:
        tree_cur = tree.DecisionTreeClassifier(random_state=12345, max_depth=depth, ccp_alpha=ccp_alpha)
        tree_cur.fit(X_train, y_train)
        trees.append(tree_cur)

    # 4. Szukanie absolutnego zwycięzcy dla danej głębokości
    best_acc = 0.0
    best_tree_idx = 0
    f1_result = 0
    best_f1_idx = 0

    for i, tree_cur in enumerate(trees):
        # PAMIĘTAJ: Zawsze testujemy na y_test!
        pred = tree_cur.predict(X_test)
        acc = accuracy_score(y_test, pred) * 100


        if acc > best_acc:
            best_acc = acc
            best_tree_idx = i

        if f1_score(y_test, pred) > f1_result:
            f1_result = f1_score(y_test, pred)
            best_f1_idx = i

    # 5. Podsumowanie
    best_alpha = ccp_alphas[best_tree_idx]
    best_layers = trees[best_tree_idx].get_depth()

    print(f"Sukces! Najlepsze Accuracy: {best_acc:.2f}% | ccp_alpha: {best_alpha:.6f} | Warstwy (layers): {best_layers}")

    print(f"Sukces! Najlepsze F1: {f1_result*100:.2f}% | ccp_alpha: {ccp_alphas[best_f1_idx]:.6f} | Warstwy (layers): {trees[best_f1_idx].get_depth()}")




--- Rozpoczynam test dla max_depth = 5 ---
Sukces! Najlepsze Accuracy: 78.92% | ccp_alpha: 0.000000 | Warstwy (layers): 5
Sukces! Najlepsze F1: 53.06% | ccp_alpha: 0.000000 | Warstwy (layers): 5

--- Rozpoczynam test dla max_depth = 6 ---
Sukces! Najlepsze Accuracy: 79.53% | ccp_alpha: 0.000025 | Warstwy (layers): 6
Sukces! Najlepsze F1: 54.16% | ccp_alpha: 0.000142 | Warstwy (layers): 6

--- Rozpoczynam test dla max_depth = 7 ---
Sukces! Najlepsze Accuracy: 79.69% | ccp_alpha: 0.000072 | Warstwy (layers): 7
Sukces! Najlepsze F1: 53.54% | ccp_alpha: 0.000025 | Warstwy (layers): 7

--- Rozpoczynam test dla max_depth = 8 ---
Sukces! Najlepsze Accuracy: 79.90% | ccp_alpha: 0.000013 | Warstwy (layers): 8
Sukces! Najlepsze F1: 54.87% | ccp_alpha: 0.000013 | Warstwy (layers): 8

--- Rozpoczynam test dla max_depth = 9 ---
Sukces! Najlepsze Accuracy: 80.20% | ccp_alpha: 0.000072 | Warstwy (layers): 9
Sukces! Najlepsze F1: 56.63% | ccp_alpha: 0.000037 | Warstwy (layers): 9

--- Rozpoczynam tes